# 🌬️ Entrenamiento del modelo eólico

**Silvana Bamba · Proyecto final de la Unidad 4**  
*CRISP-ML(Q) · Ingeniería de modelos de Machine Learning*

Comparo una referencia con un modelo de regresión para estimar la potencia activa de la turbina en kW.

## 1. Carga y separación por fecha

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

rutas = (Path("datos/T1.csv"), Path("ciencia-datos/datos/T1.csv"))
ruta_csv = next((ruta for ruta in rutas if ruta.is_file()), None)
if ruta_csv is None:
    raise FileNotFoundError("Coloca T1.csv en ciencia-datos/datos/.")

objetivo = "LV ActivePower (kW)"
entradas = ["Wind Speed (m/s)", "Wind Direction (°)"]
columna_fecha = "Date/Time"

datos = pd.read_csv(ruta_csv)
datos[columna_fecha] = pd.to_datetime(
    datos[columna_fecha], format="%d %m %Y %H:%M", errors="raise"
)
datos = datos.sort_values(columna_fecha).reset_index(drop=True)

# Mismo corte temporal del notebook 01.
fecha_corte = pd.Timestamp("2018-10-18 22:40:00")
desarrollo = datos.loc[datos[columna_fecha] < fecha_corte]
prueba_final = datos.loc[datos[columna_fecha] >= fecha_corte]
if desarrollo.empty or prueba_final.empty:
    raise ValueError("El CSV debe contener registros a ambos lados del corte temporal.")

print(f"Desarrollo: {len(desarrollo):,} | Prueba final: {len(prueba_final):,}")

Desarrollo: 40,424 | Prueba final: 10,106


## 2. Comparación en validación

Uso velocidad y dirección del viento, las dos entradas del formulario. La fecha sirve para ordenar los datos. El último 20 % del tramo de desarrollo se usa para elegir el modelo por MAE.

In [2]:
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

corte_validacion = int(len(desarrollo) * 0.8)
ajuste = desarrollo.iloc[:corte_validacion]
validacion = desarrollo.iloc[corte_validacion:]

modelos = {
    "Referencia (mediana)": DummyRegressor(strategy="median"),
    "Gradient boosting": HistGradientBoostingRegressor(
        max_iter=150, early_stopping=False, random_state=42
    ),
}

def calcular_metricas(reales, estimadas):
    return {
        "MAE (kW)": float(mean_absolute_error(reales, estimadas)),
        "RMSE (kW)": float(mean_squared_error(reales, estimadas) ** 0.5),
        "R²": float(r2_score(reales, estimadas)),
    }

resultados = {}
for nombre, modelo in modelos.items():
    modelo.fit(ajuste[entradas], ajuste[objetivo])
    estimadas = modelo.predict(validacion[entradas])
    resultados[nombre] = calcular_metricas(validacion[objetivo], estimadas)

tabla_validacion = pd.DataFrame.from_dict(resultados, orient="index")
tabla_validacion = tabla_validacion.sort_values("MAE (kW)", kind="stable")
display(tabla_validacion.round(3))

mae_referencia = resultados["Referencia (mediana)"]["MAE (kW)"]
mae_candidato = resultados["Gradient boosting"]["MAE (kW)"]
print(f"Reducción del MAE frente a la referencia: {mae_referencia - mae_candidato:.2f} kW")

,MAE (kW),RMSE (kW),R²
Gradient boosting,139.710,291.145,0.948
Referencia (mediana),1215.502,1561.004,-0.482


Reducción del MAE frente a la referencia: 1075.79 kW


## 3. Evaluación final y exportación

Ajusto el modelo elegido con todo el tramo de desarrollo y mido su error en los registros reservados. Guardo el modelo, sus entradas y sus métricas para usarlo en la API.

In [3]:
import joblib
import sklearn

nombre_elegido = tabla_validacion.index[0]
modelo_final = clone(modelos[nombre_elegido])
modelo_final.fit(desarrollo[entradas], desarrollo[objetivo])

predicciones = modelo_final.predict(prueba_final[entradas])
metricas_finales = calcular_metricas(prueba_final[objetivo], predicciones)

print(f"Modelo elegido por MAE de validación: {nombre_elegido}")
display(pd.DataFrame([metricas_finales], index=[nombre_elegido]).round(3))

# La salida es estable aunque T1.csv se lea desde cualquiera de las rutas admitidas.
ruta_modelo = Path("ciencia-datos/artefactos/modelo_turbina.joblib")
ruta_modelo.parent.mkdir(parents=True, exist_ok=True)

joblib.dump({
    "modelo": modelo_final,
    "nombre_modelo": nombre_elegido,
    "entradas": entradas,
    "objetivo": objetivo,
    "metricas_prueba": metricas_finales,
    "fecha_corte": str(fecha_corte),
    "version_sklearn": sklearn.__version__,
}, ruta_modelo)

artefacto = joblib.load(ruta_modelo)
ejemplo = artefacto["modelo"].predict(prueba_final[artefacto["entradas"]].head(1))
print(f"Modelo guardado en: {ruta_modelo}")
print(f"Predicción de comprobación: {ejemplo[0]:.2f} kW")

Modelo elegido por MAE de validación: Gradient boosting


,MAE (kW),RMSE (kW),R²
Gradient boosting,250.438,517.004,0.852


Modelo guardado en: artefactos\modelo_turbina.joblib
Predicción de comprobación: 2.34 kW


## Resultado

Elegí Gradient boosting porque redujo el MAE de 1.215,50 a 139,71 kW
en validación. En la prueba final, el MAE fue de 250,44 kW.
Exporté el modelo para usarlo en la API.